In [115]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import wkt
from rapidfuzz import fuzz, process
from tqdm import tqdm

In [116]:
def lists_overlap(a, b):
    # Handle missing values (NaN) in the cells themselves
    if not isinstance(a, list) or not isinstance(b, list):
        return False
    if a == b:
        return True
    else: 
        return len(set(a) & set(b)) > 0



In [117]:
def add_appendix_to_col(col_names, appendix):
    new_cols = {}
    for col in col_names:
        new_cols[col]=f'{col}_appendix'
    return new_cols
    

In [118]:
gavoc_overview = pd.read_pickle('../data/gavoc-overview.pkl')

In [119]:
gavoc_overview = gpd.GeoDataFrame(gavoc_overview)
coord_strings = gavoc_overview["point_coord"]
gavoc_overview["point_coord"] = gpd.GeoSeries.from_wkt(coord_strings)
gavoc_overview = gavoc_overview.set_geometry("point_coord")
gavoc_overview = gavoc_overview.set_crs('3857')

In [120]:
# load glob_places
dfs = pd.read_excel('../places_20260709.xlsx', sheet_name=None)

In [121]:
glob_places = pd.merge(
    dfs['labels'][['glob_id', 'label']],
    dfs['places'][['glob_id', 'pref_label', 'latitude', 'longitude', 'wkt']],
    how='left',
    on='glob_id'
)

In [122]:
glob_places["wkt"] = glob_places["wkt"].fillna("POINT EMPTY")

In [123]:
glob_places = gpd.GeoDataFrame(glob_places)
coord_strings = glob_places["wkt"]
glob_places["wkt"] = gpd.GeoSeries.from_wkt(coord_strings)
glob_places = glob_places.set_geometry("wkt")
glob_places = glob_places.set_crs('3857')

In [124]:
glob_types = dfs['types'][['glob_id', 'place_type_uri', 'place_type']].copy()

In [125]:
gavoc_overview.iloc[0]

external_id                                           GLOB_GAVOC_00001
pref_label                                             Andaman Islands
label                Andamao|Aandomaon, Ilha de|Andamaon|Andaman Is...
longitude                                                    92.833333
latitude                                                          12.5
point_coord                                     POINT (92.833333 12.5)
coord_source         https://www.zotero.org/groups/4678659/items/HR...
coord_source_page                                                  399
coord_remarks                                            12-30N/92-50E
place_type                                                 archipelago
pp_uri               https://digitaalerfgoed.poolparty.biz/globalis...
pp_type_label                                              archipelago
attestation_id                                               2|307|308
Name: 0, dtype: object

In [126]:
fuzz.WRatio('adamanen', 'andaman eilanden')

72.0

In [127]:
def find_match(label, candidates, precision=82, cols=['glob_id','pref_label','label'], candidate_col='label'):
    candidates = candidates[cols]
    
    matches = process.extract(
    label,
    candidates[candidate_col],
    scorer=fuzz.ratio,
    limit=5,
    score_cutoff=precision
    )
    if len(matches) == 0:
        return None

    # Extract indices and scores
    top_indices = [match[2] for match in matches]
    scores = [match[1] for match in matches]
    
    # Select the rows and add similarity score
    top_rows = candidates.iloc[top_indices].copy()
    top_rows['similarity_score'] = scores
    
    # Sort by similarity score descending
    top_rows = top_rows.sort_values(by='similarity_score', ascending=False)
    
    return top_rows

In [128]:
def pare_down_matches(matched_rows, groupby_col='glob_id'):
    if len(matched_rows) < 2:
        return matched_rows
    
    return matched_rows.groupby(groupby_col).max().reset_index()


In [129]:
def add_match_metadata(matched_rows, method, external_id):
    matched_rows['method'] = method
    matched_rows['matched_id'] = external_id
    return matched_rows

In [ ]:
def match_external_to_glob(df, glob_places=glob_places, distance=5, precision=82, no_coords_skip=True, precision_penalty=90):
    matched_results = []
    passed_rows = []
    for i, row in tqdm(df.iterrows(), total=df.shape[0]):
        ref_point = row['point_coord']   
        if not ref_point.is_empty:
            matchable_options = glob_places[glob_places.geometry.distance(ref_point) <= distance].reset_index().copy()
            candidate_matches = [find_match(label, matchable_options, precision) for label in row['label'].split('|')]
            candidate_matches = [i for i in candidate_matches if i is not None]
            if len(candidate_matches)>0:
                candidate_matches = pd.concat(candidate_matches)
                candidate_matches = pare_down_matches(candidate_matches)
                candidate_matches = add_match_metadata(
                    candidate_matches,
                    f'{distance}',
                    row['external_id'],
                )
                matched_results.append(candidate_matches)
            else:
                passed_rows.append(row)
        elif no_coords_skip:
            passed_rows.append(row)
        else:
            matchable_options = glob_places.copy()
            candidate_matches = [find_match(label, matchable_options, precision_penalty) for label in row['label'].split('|')]
            candidate_matches = [i for i in candidate_matches if i is not None]
            if len(candidate_matches)>0:
                candidate_matches = pd.concat(candidate_matches)
                candidate_matches = pare_down_matches(candidate_matches)
                candidate_matches = add_match_metadata(
                    candidate_matches,
                    f'no_coordinates',
                    row['external_id'],
                )
                matched_results.append(candidate_matches)

    return pd.concat(matched_results), pd.DataFrame(passed_rows)


In [131]:
def match_with_increasing_dist(df, distance=5, distance_increases=1, iterations=5, no_coords_skip=True):
    matched_results = []
    for iteration in range(iterations):
        if iteration > 1:
            no_coords_skip = True 
        print(f"running round {iteration+1} of {iterations}")
        matches, df = match_external_to_glob(df, distance=distance, no_coords_skip=no_coords_skip)
        matched_results.append(matches)
        distance += distance_increases
        print(f"new matches: {len(matches)}\nunmatched remaining: {len(df)}\n")
    f"\nComplete."
    return pd.concat(matched_results), df
    

In [132]:
def find_internal_match(df, distance=1, no_coords_skip=True, precision=85):
    matched_results = []
    passed_rows = []
    for i, row in tqdm(df.iterrows(), total=df.shape[0]):
        ref_point = row['point_coord']  
        if not ref_point.is_empty:
            matchable_options = df[
                (df.geometry.distance(ref_point) <= distance) & 
                (df.external_id != row.external_id) &
                (df.pp_type_label == row.pp_type_label)
                ].reset_index().copy()
            candidate_matches = [
                find_match(label, 
                           matchable_options, 
                           precision=precision,
                           cols=['external_id','pref_label','label'], 
                           candidate_col='pref_label'
                           ) for label in row['label'].split('|')]
            candidate_matches = [i for i in candidate_matches if i is not None]
            if len(candidate_matches)>0:
                candidate_matches = pd.concat(candidate_matches)
                candidate_matches = pare_down_matches(candidate_matches, 'external_id')
                candidate_matches = add_match_metadata(
                    candidate_matches,
                    f'{distance}',
                    row['external_id'],
                )
                matched_results.append(candidate_matches)
            else:
                passed_rows.append(row)
        else:
            passed_rows.append(row)
    return pd.concat(matched_results), pd.DataFrame(passed_rows)

In [133]:
internal_matches, unmatched = find_internal_match(gavoc_overview)

100%|██████████| 8638/8638 [00:08<00:00, 989.32it/s]


In [134]:
def combine_internal_matches(internal_matches, df):
    matched_list = []
    internal_matches = internal_matches[['external_id', 'matched_id']].groupby('external_id').agg(lambda col: list(set(col))).reset_index()

    df = pd.merge(df, internal_matches, how='left', on='external_id')
    df['matched_id'] = df['matched_id'].fillna('-')
    df['group_by_id'] = list(df[['external_id', 'matched_id']].apply(
        lambda row: sorted([row.external_id] + row.matched_id) if row.matched_id != '-' else [row.external_id], axis=1)
        )
    df['group_by_id'] = df['group_by_id'].apply(lambda x: '|'.join(x))
    df = df.drop('matched_id', axis=1)
    df = df.groupby('group_by_id').agg(lambda col: list(set(col))).reset_index()

    return df

In [139]:
# x = combine_internal_matches(internal_matches, gavoc_overview)
x[x.external_id.apply(lambda x: len(x))>1]

,group_by_id,external_id,pref_label,label,longitude,latitude,point_coord,coord_source,coord_source_page,coord_remarks,place_type,pp_uri,pp_type_label,attestation_id
18,GLOB_GAVOC_00036|GLOB_GAVOC_00037,"[GLOB_GAVOC_00037, GLOB_GAVOC_00036]",[Adan],"[Oud-Aden|Adan|Ade Vechie, Adan|Adon|Aden]","[45.016667, 44.866667]","[12.733333, 12.8]","[POINT (45.016667 12.8), POINT (44.866667 12.7...",[https://www.zotero.org/groups/4678659/items/H...,[399],"[12-48N/45-01E, 12-44N/44-52E]",[settlement],[https://digitaalerfgoed.poolparty.biz/globali...,[settlements],"[37, 49|38|39]"
23,GLOB_GAVOC_00046|GLOB_GAVOC_00182,"[GLOB_GAVOC_00182, GLOB_GAVOC_00046]","[Alluru, Allur]","[Allur|Adoer, Alluru|Aloer]","[80.133333, 80.05]","[14.666667, 15.5]","[POINT (80.133333 15.5), POINT (80.05 14.666667)]",[https://www.zotero.org/groups/4678659/items/H...,[399],"[14-40N/80-03E, 15-30N/80-08E]",[settlement],[https://digitaalerfgoed.poolparty.biz/globali...,[settlements],"[47, 183]"
36,GLOB_GAVOC_00082|GLOB_GAVOC_00083,"[GLOB_GAVOC_00082, GLOB_GAVOC_00083]",[Agra],"[Agra, Logie|Agra, Agra]",[78.016667],[27.166667],[POINT (78.016667 27.166667)],[https://www.zotero.org/groups/4678659/items/H...,[399],[27-10N/78-01E],[settlement],[https://digitaalerfgoed.poolparty.biz/globali...,[settlements],"[84, 83]"
56,GLOB_GAVOC_00111|GLOB_GAVOC_00455,"[GLOB_GAVOC_00111, GLOB_GAVOC_00455]",[Ammapattinam],"[Ammapattinam|Aitapatnam, Aripagoedy|Ammapatti...",[79.233333],"[10.033333, 10.016667]","[POINT (79.233333 10.033333), POINT (79.233333...",[https://www.zotero.org/groups/4678659/items/H...,[399],"[10-01N/79-14E, 10-02N/79-14E]",[settlement],[https://digitaalerfgoed.poolparty.biz/globali...,[settlements],"[456, 112]"
65,GLOB_GAVOC_00128|GLOB_GAVOC_00145,"[GLOB_GAVOC_00145, GLOB_GAVOC_00128]",[Alantalai],"[Alantalai|Alendum, Alandale|Alantalai]","[78.1, 78.166667]",[8.466667],"[POINT (78.1 8.466667), POINT (78.166667 8.466...",[https://www.zotero.org/groups/4678659/items/H...,[399],"[08-28N/78-10E, 08-28N/78-06E]",[settlement],[https://digitaalerfgoed.poolparty.biz/globali...,[settlements],"[129, 146]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4736,GLOB_GAVOC_10752|GLOB_GAVOC_10754|GLOB_GAVOC_1...,"[GLOB_GAVOC_10752, GLOB_GAVOC_10808, GLOB_GAVO...",[Vedalai],"[Vedalai|Wedale, Vedalai|Wetale]","[79.108333, 79.091667]","[9.266667, 9.283333]","[POINT (79.108333 9.266667), POINT (79.091667 ...",[https://www.zotero.org/groups/4678659/items/H...,[416],"[09-16N/79-05-30E, 09-17N/79-05-30E, 09-16N/79...",[settlement],[https://digitaalerfgoed.poolparty.biz/globali...,[settlements],"[10753|10754, 10809, 10755]"
4742,GLOB_GAVOC_10761|GLOB_GAVOC_10817,"[GLOB_GAVOC_10817, GLOB_GAVOC_10761]",[Velanai East],"[Welene|Velanai East, Weyelehaly|Velanai East]","[79.883333, 79.9]",[9.633333],"[POINT (79.9 9.633333), POINT (79.883333 9.633...",[https://www.zotero.org/groups/4678659/items/H...,[416],"[09-38N/79-53E, 09-38N/79-54E]",[settlement],[https://digitaalerfgoed.poolparty.biz/globali...,[settlements],"[10762, 10818]"
4787,GLOB_GAVOC_10885|GLOB_GAVOC_10886,"[GLOB_GAVOC_10886, GLOB_GAVOC_10885]",[Wosi],"[Wosi|Wosie|Wosse, Negery, Wosi|Wosje, Negery]","[127.95, 127.966667]","[-0.166667, -0.183333]","[POINT (127.95 -0.166667), POINT (127.966667 -...",[https://www.zotero.org/groups/4678659/items/H...,[416],"[00-11S/127-58E, 00-10S/127-57E]",[settlement],[https://digitaalerfgoed.poolparty.biz/globali...,[settlements],"[10888|10886, 10887]"
4818,GLOB_GAVOC_10962|GLOB_GAVOC_10981,"[GLOB_GAVOC_10981, GLOB_GAVOC_10962]",[Samau],"[Samau|Zemou, Po.|Semaoe, Samau|Semaoe|Zemau|Z...","[123.383333, 123.366667]","[-10.25, -10.216667]","[POINT (123.383333 -10.25), POINT (123.366667 ...",[https://www.zotero.org/groups/4678659/items/H...,"[417, 416|417]","[10-13S/123-22E, 10-15S/123-23E]",[island],[https://digitaalerfgoed.poolparty.biz/globali...,[islands],"[10963|10981|10979, 10982]"


In [136]:
internal_matches

,external_id,pref_label,label,similarity_score,method,matched_id
0,GLOB_GAVOC_00037,Adan,Adan|Adon|Aden,100.000000,1,GLOB_GAVOC_00036
0,GLOB_GAVOC_00036,Adan,Oud-Aden|Adan|Ade Vechie,100.000000,1,GLOB_GAVOC_00037
0,GLOB_GAVOC_00183,Alluru,Alluru|Aloer,90.909091,1,GLOB_GAVOC_00046
0,GLOB_GAVOC_00083,Agra,"Agra, Logie|Agra",100.000000,1,GLOB_GAVOC_00082
0,GLOB_GAVOC_00082,Agra,Agra,100.000000,1,GLOB_GAVOC_00083
...,...,...,...,...,...,...
1,GLOB_GAVOC_11082,Saylac,Zeila|Saylac|Zeylá,100.000000,1,GLOB_GAVOC_11061
10,GLOB_GAVOC_11051,Samau,"Samau|Semaoe|Zemau|Zemanto, Po.|Zeemou",100.000000,1,GLOB_GAVOC_11070
99,GLOB_GAVOC_09023,Sendalaipattanam,Sindale|Sendalaipattanam,100.000000,1,GLOB_GAVOC_11072
0,GLOB_GAVOC_11061,Saylac,Zeylaas Cust|Saylac|Zeila|Zeilán,100.000000,1,GLOB_GAVOC_11082


In [110]:
matched_pairs = gavoc_matched[[
    'external_id',
    'glob_id',
    'pref_label',
    'similarity_score',
    'method',
]].copy()

In [111]:
glob_types = glob_types.fillna('-')

In [112]:
def extend_matches(glob_types, gavoc_overview, matched_results):
    # join the dataframes
    glob_types = glob_types.groupby('glob_id').agg(lambda col: list(set(col))).map(lambda x: '|'.join(x)).reset_index()
    matched_results = pd.merge(gavoc_overview, matched_results, how='right',on='external_id')
    matched_results = pd.merge(matched_results, glob_types, how='left',on='glob_id')
    matched_results['pp_uri'] = matched_results['pp_uri']
    matched_results['overlap'] = matched_results['pp_uri'] == matched_results['place_type_uri']

    return matched_results[['glob_id', 'external_id', 'similarity_score', 'method', 'overlap']]
    

In [113]:
df = extend_matches(glob_types, gavoc_overview, gavoc_matched)

In [114]:
glob_overview = pd.merge(glob_places, glob_types, how='left', on='glob_id')
glob_overview = glob_overview.fillna('-')
glob_overview['latitude'] = glob_overview['latitude'].apply(lambda x: f'{x}')
glob_overview['longitude'] = glob_overview['longitude'].apply(lambda x: f'{x}')
glob_overview['wkt'] = glob_overview['wkt'].apply(lambda x: f'{x}')
glob_overview = glob_overview.groupby('glob_id').agg(lambda col: list(set(col))).map(lambda x: '|'.join(x)).reset_index()

/var/folders/lk/z05v_dms06ndgslfn2ww0bkmfbw0qd/T/ipykernel_71967/953269564.py:5: UserWarning: Geometry column does not contain geometry.
  glob_overview['wkt'] = glob_overview['wkt'].apply(lambda x: f'{x}')


In [115]:
df.to_csv('gavoc_matches.csv', index=False)

In [116]:
glob_overview.to_csv('glob_overview.csv', index=False)

In [117]:
gavoc_overview.to_csv('gavoc_overview.csv', index=False)

In [118]:
df['overlap'].sum()

np.int64(193)

## Match to GEONAMES

In [119]:
unmatched

,external_id,pref_label,label,longitude,latitude,point_coord,coord_source,coord_source_page,coord_remarks,place_type,pp_uri,pp_type_label,attestation_id
1,GLOB_GAVOC_00002,Bukit Monyet,Apenberg|Aapjesberg|Bukit Monyet,100.333333,-0.966667,POINT (100.333333 -0.966667),https://www.zotero.org/groups/4678659/items/HR...,399,00-58S/100-20E,mountain,https://digitaalerfgoed.poolparty.biz/globalis...,mountains,3
2,GLOB_GAVOC_00003,Arnhem Land,Aarnhems Landt|Arnhem Land|Aarnhems Land,134.0,-13.5,POINT (134 -13.5),https://www.zotero.org/groups/4678659/items/HR...,399,13-30S/134E,region,https://digitaalerfgoed.poolparty.biz/globalis...,regions,4|5
3,GLOB_GAVOC_00005,Tg. Mangkalihat,Aart Gysens H.|art Gysens Hoek|Tg. Mangkalihat,119.0,1.0,POINT (119 1),https://www.zotero.org/groups/4678659/items/HR...,399,01N/119E,cape,https://digitaalerfgoed.poolparty.biz/globalis...,capes,6|7
4,GLOB_GAVOC_00007,Per Aru,"Aatchellay, R.|Per Aru",80.75,9.25,POINT (80.75 9.25),https://www.zotero.org/groups/4678659/items/HR...,399,09-15N/80-45E,river,https://digitaalerfgoed.poolparty.biz/globalis...,rivers,8
5,GLOB_GAVOC_00011,Port Mc Arthur,Abel Tasmans Bayen|Port Mc Arthur,136.75,-15.833333,POINT (136.75 -15.833333),https://www.zotero.org/groups/4678659/items/HR...,399,15-50S/136-45E,bay,https://digitaalerfgoed.poolparty.biz/globalis...,bays,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5105,GLOB_GAVOC_11074,South Entrance to Grand Port,"Zuyd-Ooster Gat, C. t|South Entrance to Grand ...",57.766667,-20.4,POINT (57.766667 -20.4),https://www.zotero.org/groups/4678659/items/HR...,417,20-24S/57-46E,cape,https://digitaalerfgoed.poolparty.biz/globalis...,capes,11075
5106,GLOB_GAVOC_11075,South Entrance to Grand Port,"Zuyd-Ooster Gat, t|South Entrance to Grand Port",57.766667,-20.4,POINT (57.766667 -20.4),https://www.zotero.org/groups/4678659/items/HR...,417,20-24S/57-46E,strait,https://digitaalerfgoed.poolparty.biz/globalis...,straits,11076
5107,GLOB_GAVOC_11078,Tg. Lombe,Tg. Lombe|Z.W. Hoek,122.566667,-5.516667,POINT (122.566667 -5.516667),https://www.zotero.org/groups/4678659/items/HR...,417,05-31S/122-34E,cape,https://digitaalerfgoed.poolparty.biz/globalis...,capes,11079
5108,GLOB_GAVOC_11079,Noma Misaki,Z.W. Hoek|Noma Misaki,130.1,31.4,POINT (130.1 31.4),https://www.zotero.org/groups/4678659/items/HR...,417,31-24N/130-06E,cape,https://digitaalerfgoed.poolparty.biz/globalis...,capes,11080
